# Multivariate marked Hawkes–GPD model for extreme loss clustering and spillover in Vietnamese equities

Notebook này xây dựng pipeline thực nghiệm cho 7 cổ phiếu Việt Nam: `SSI`, `TCB`, `MBB`, `VPB`, `HPG`, `VHM`, `FPT` với ticker Yahoo Finance tương ứng `SSI.VN`, `TCB.VN`, `MBB.VN`, `VPB.VN`, `HPG.VN`, `VHM.VN`, `FPT.VN` trong giai đoạn **2019-01-01 → 2026-06-30**.

Thứ tự mô hình:

1. Historical VaR/ES
2. Poisson–GPD
3. Hawkes–GPD
4. Marked Hawkes–GPD
5. Multivariate Hawkes–GPD
6. Multivariate marked Hawkes–GPD

Nguyên tắc tiết kiệm mô hình: **không nâng cấp lên mô hình phức tạp nếu mô hình đơn giản đã đủ tốt**. Quyết định này được thực hiện bởi `should_upgrade_model()` dựa trên backtesting, AIC/BIC, clustering và spillover.

In [ ]:
# Nếu chạy trên môi trường mới, có thể cần cài các gói sau:
# %pip install yfinance pandas numpy scipy matplotlib seaborn nbformat reportlab joblib

import json
import math
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import yfinance as yf
except Exception as exc:
    yf = None
    print("yfinance chưa sẵn sàng:", exc)

from scipy.optimize import minimize
from scipy.stats import chi2, genpareto, binomtest
from scipy.linalg import eigvals
import joblib

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet
    REPORTLAB_AVAILABLE = True
except Exception:
    REPORTLAB_AVAILABLE = False

ROOT = Path.cwd()
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"
OUT_MODELS = ROOT / "outputs" / "models"
OUT_REPORTS = ROOT / "outputs" / "reports"
for p in [OUT_TABLES, OUT_FIGURES, OUT_MODELS, OUT_REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

ASSETS = ["SSI", "TCB", "MBB", "VPB", "HPG", "VHM", "FPT"]
TICKERS = [f"{x}.VN" for x in ASSETS]
TICKER_TO_ASSET = dict(zip(TICKERS, ASSETS))
START_DATE = "2019-01-01"
END_DATE = "2026-06-30"
TRAIN_END = "2024-12-31"
TEST_START = "2025-01-01"
Q_MAIN = 0.95
ALPHAS = [0.95, 0.975, 0.99]
TRADING_DAYS = 252
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Tải dữ liệu, kiểm tra chất lượng, tính log-return và loss

Dùng Adjusted Close nếu Yahoo Finance trả về cột `Adj Close`; nếu `auto_adjust=True` trả về `Close` đã điều chỉnh thì notebook tự dùng cột đó. Loss được định nghĩa là \(L_{k,j}=-r_{k,j}\\).

In [ ]:
def download_prices(tickers: List[str], start: str, end: str) -> pd.DataFrame:
    if yf is None:
        raise ImportError("Cần cài yfinance để tải dữ liệu Yahoo Finance.")
    raw = yf.download(tickers, start=start, end=pd.Timestamp(end) + pd.Timedelta(days=1),
                      auto_adjust=False, progress=False, group_by="column")
    if isinstance(raw.columns, pd.MultiIndex):
        if "Adj Close" in raw.columns.get_level_values(0):
            px = raw["Adj Close"].copy()
        elif "Close" in raw.columns.get_level_values(0):
            px = raw["Close"].copy()
        else:
            raise ValueError("Không tìm thấy Adj Close/Close trong dữ liệu tải về.")
    else:
        col = "Adj Close" if "Adj Close" in raw.columns else "Close"
        px = raw[[col]].rename(columns={col: tickers[0]})
    px = px.rename(columns=TICKER_TO_ASSET).sort_index()
    px = px[ASSETS]
    return px

def compute_quality_metrics(prices: pd.DataFrame, returns: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for a in ASSETS:
        s = prices[a]
        r = returns[a]
        positive = s.dropna()
        floor = positive.min() if len(positive) else np.nan
        rows.append({
            "asset": a,
            "missing_rate": float(s.isna().mean()),
            "zero_return_rate": float((r.dropna() == 0).mean()) if r.notna().any() else np.nan,
            "floor_price": float(floor) if pd.notna(floor) else np.nan,
            "floor_hit_rate": float((s == floor).mean()) if pd.notna(floor) else np.nan,
            "n_prices": int(s.notna().sum()),
            "n_returns": int(r.notna().sum()),
        })
    return pd.DataFrame(rows)

prices = download_prices(TICKERS, START_DATE, END_DATE)
log_prices = np.log(prices)
returns = log_prices.diff().dropna(how="all")
losses = -returns

quality = compute_quality_metrics(prices, returns)
quality.to_csv(OUT_TABLES / "data_quality.csv", index=False)
prices.to_csv(OUT_TABLES / "adjusted_close.csv")
returns.to_csv(OUT_TABLES / "log_returns.csv")
losses.to_csv(OUT_TABLES / "losses.csv")

train_losses = losses.loc[:TRAIN_END].copy()
test_losses = losses.loc[TEST_START:END_DATE].copy()
print(prices.tail())
display(quality)

In [ ]:
def savefig(name: str):
    png = OUT_FIGURES / f"{name}.png"
    pdf = OUT_FIGURES / f"{name}.pdf"
    plt.tight_layout()
    plt.savefig(png, dpi=180, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(12, 5))
(prices / prices.iloc[0]).plot(ax=plt.gca(), lw=1.2)
plt.title("Adjusted close normalized to 1.0")
plt.ylabel("Normalized price")
savefig("normalized_adjusted_close")

plt.figure(figsize=(12, 5))
losses.plot(ax=plt.gca(), lw=0.7, alpha=0.85)
plt.title("Daily losses: -log returns")
plt.ylabel("Loss")
savefig("daily_losses")

**Diễn giải dữ liệu.** Missing rate cao có thể làm lệch ước lượng xác suất cực trị; zero-return rate và floor-hit rate cao thường phản ánh thanh khoản thấp, biên độ giá hoặc lỗi dữ liệu. Các bảng/hình đã được lưu trong `outputs/tables` và `outputs/figures`.

## 2. Hàm EVT/GPD, backtesting, likelihood và quyết định nâng cấp mô hình

In [ ]:
def gpd_fit(excess: np.ndarray) -> Dict[str, float]:
    y = np.asarray(excess, dtype=float)
    y = y[np.isfinite(y) & (y > 0)]
    if len(y) < 10:
        return {"xi": np.nan, "beta": np.nan, "ll": np.nan}
    c, loc, scale = genpareto.fit(y, floc=0)
    ll = float(np.sum(genpareto.logpdf(y, c=c, loc=0, scale=scale)))
    return {"xi": float(c), "beta": float(scale), "ll": ll}

def evt_var_es(u: float, xi: float, beta: float, T: int, n_u: int, alpha: float) -> Tuple[float, float]:
    if n_u <= 0 or T <= 0 or not np.isfinite([u, xi, beta]).all() or beta <= 0:
        return np.nan, np.nan
    p = (T / n_u) * (1 - alpha)
    if abs(xi) < 1e-8:
        var = u - beta * np.log(p)
    else:
        var = u + (beta / xi) * (p ** (-xi) - 1)
    es = np.nan if xi >= 1 else (var + beta - xi * u) / (1 - xi)
    return float(var), float(es)

def historical_var_es(train: pd.Series, alpha: float) -> Tuple[float, float]:
    x = train.dropna().values
    var = float(np.quantile(x, alpha))
    tail = x[x > var]
    es = float(tail.mean()) if len(tail) else var
    return var, es

def kupiec_test(violations: np.ndarray, alpha: float) -> Dict[str, float]:
    v = np.asarray(violations, dtype=int)
    n = len(v); x = int(v.sum()); p0 = 1 - alpha
    if n == 0:
        return {"n": 0, "violations": 0, "violation_rate": np.nan, "kupiec_lr": np.nan, "kupiec_pvalue": np.nan}
    phat = min(max(x / n, 1e-12), 1 - 1e-12)
    p0c = min(max(p0, 1e-12), 1 - 1e-12)
    ll0 = (n - x) * np.log(1 - p0c) + x * np.log(p0c)
    ll1 = (n - x) * np.log(1 - phat) + x * np.log(phat)
    lr = max(0.0, -2 * (ll0 - ll1))
    return {"n": n, "violations": x, "violation_rate": x / n, "kupiec_lr": lr, "kupiec_pvalue": float(1 - chi2.cdf(lr, 1))}

def information_criteria(loglik: float, n_params: int, n_obs: int) -> Tuple[float, float]:
    if not np.isfinite(loglik) or n_obs <= 0:
        return np.nan, np.nan
    return 2 * n_params - 2 * loglik, np.log(n_obs) * n_params - 2 * loglik

def should_upgrade_model(current: Dict, candidate_context: Dict, min_aic_improvement: float = 2.0) -> Tuple[bool, str]:
    """Quyết định có nâng cấp mô hình không.

    Nâng cấp nếu một trong các điều kiện thực nghiệm quan trọng xuất hiện:
    - Kupiec p-value thấp hoặc violation rate lệch đáng kể so với mức kỳ vọng.
    - AIC/BIC của mô hình phức tạp cải thiện đủ lớn.
    - Có dấu hiệu clustering: branching ratio cao, hoặc autocorrelation indicator cực trị cao.
    - Có dấu hiệu spillover: spectral radius/ma trận nhánh đa biến không nhỏ.
    """
    alpha = current.get("alpha", 0.99)
    expected = 1 - alpha
    vr = current.get("violation_rate", np.nan)
    kupiec_p = current.get("kupiec_pvalue", np.nan)
    aic = current.get("aic", np.nan)
    cand_aic = candidate_context.get("candidate_aic", np.nan)
    clustering = candidate_context.get("clustering_score", 0.0)
    spillover = candidate_context.get("spillover_score", 0.0)

    reasons = []
    if np.isfinite(kupiec_p) and kupiec_p < 0.05:
        reasons.append("Kupiec p-value < 0.05")
    if np.isfinite(vr) and abs(vr - expected) > max(0.01, expected):
        reasons.append("violation rate lệch đáng kể")
    if np.isfinite(aic) and np.isfinite(cand_aic) and (aic - cand_aic) >= min_aic_improvement:
        reasons.append("AIC cải thiện đủ lớn")
    if clustering > 0.10:
        reasons.append("có clustering cực trị")
    if spillover > 0.05:
        reasons.append("có spillover đáng kể")
    if reasons:
        return True, "; ".join(reasons)
    return False, "Mô hình hiện tại đủ tốt theo Kupiec/violation/AIC và tín hiệu clustering-spillover yếu"

## 3. Historical VaR/ES

Mô hình lịch sử không giả định phân phối tham số. Đây là baseline để kiểm tra VaR/ES trên test set.

In [ ]:
model_rows = []
for a in ASSETS:
    for alpha in ALPHAS:
        var, es = historical_var_es(train_losses[a], alpha)
        kt = kupiec_test((test_losses[a].dropna().values > var).astype(int), alpha)
        row = {"model": "Historical", "asset": a, "alpha": alpha, "VaR": var, "ES": es,
               "log_likelihood": np.nan, "AIC": np.nan, "BIC": np.nan, "xi": np.nan, "beta_gpd": np.nan,
               "threshold": np.nan, "n_exceed": np.nan, **kt}
        model_rows.append(row)
historical_results = pd.DataFrame(model_rows)
historical_results.to_csv(OUT_TABLES / "historical_var_es_backtest.csv", index=False)
display(historical_results)

**Diễn giải Historical.** Nếu violation rate gần \(1-\alpha\\) và Kupiec p-value không bác bỏ, baseline có thể đủ tốt cho VaR tĩnh. Tuy nhiên Historical VaR/ES không mô tả tail index, clustering hay spillover, nên các phần sau chỉ nâng cấp khi dữ liệu cho thấy cần thiết.

## 4. Poisson–GPD

Poisson–GPD tách tần suất vượt ngưỡng và độ lớn exceedance. Ngưỡng chính là \(q=0.95\\), \(u_k=Q_k(q)\\).

In [ ]:
def fit_poisson_gpd_for_asset(asset: str, q: float = Q_MAIN) -> Dict:
    x = train_losses[asset].dropna()
    u = float(x.quantile(q))
    exc = x[x > u] - u
    g = gpd_fit(exc.values)
    T = len(x); n_u = len(exc)
    pois_rate = n_u / T if T else np.nan
    ll_pois = n_u * np.log(pois_rate + 1e-12) - T * pois_rate if T else np.nan
    ll = ll_pois + g["ll"]
    aic, bic = information_criteria(ll, 3, T)
    out = {"asset": asset, "threshold": u, "n_exceed": n_u, "T": T, "poisson_rate": pois_rate,
           "xi": g["xi"], "beta_gpd": g["beta"], "log_likelihood": ll, "AIC": aic, "BIC": bic}
    for alpha in ALPHAS:
        var, es = evt_var_es(u, g["xi"], g["beta"], T, n_u, alpha)
        kt = kupiec_test((test_losses[asset].dropna().values > var).astype(int), alpha)
        out.update({f"VaR_{alpha}": var, f"ES_{alpha}": es, f"violation_rate_{alpha}": kt["violation_rate"],
                    f"kupiec_pvalue_{alpha}": kt["kupiec_pvalue"]})
    return out

poisson_gpd = pd.DataFrame([fit_poisson_gpd_for_asset(a) for a in ASSETS])
poisson_gpd.to_csv(OUT_TABLES / "poisson_gpd_results.csv", index=False)
joblib.dump(poisson_gpd, OUT_MODELS / "poisson_gpd.joblib")
display(poisson_gpd)

In [ ]:
def threshold_sensitivity(asset: str, qs=(0.90, 0.925, 0.95, 0.975)) -> pd.DataFrame:
    rows=[]
    x = train_losses[asset].dropna()
    for q in qs:
        u = float(x.quantile(q)); exc = x[x > u] - u; g = gpd_fit(exc.values)
        row = {"asset": asset, "q": q, "threshold": u, "n_exceed": len(exc), "xi": g["xi"], "beta_gpd": g["beta"]}
        var, es = evt_var_es(u, g["xi"], g["beta"], len(x), len(exc), 0.99)
        row.update({"VaR_0.99": var, "ES_0.99": es})
        rows.append(row)
    return pd.DataFrame(rows)

sens = pd.concat([threshold_sensitivity(a) for a in ASSETS], ignore_index=True)
sens.to_csv(OUT_TABLES / "threshold_sensitivity.csv", index=False)
display(sens)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for a in ASSETS:
    sub = sens[sens.asset == a]
    axes[0].plot(sub.q, sub.xi, marker="o", label=a)
    axes[1].plot(sub.q, sub["VaR_0.99"], marker="o", label=a)
axes[0].set_title("Threshold sensitivity: tail index xi")
axes[1].set_title("Threshold sensitivity: EVT VaR 99%")
for ax in axes: ax.grid(True, alpha=.3); ax.legend(fontsize=8)
savefig("threshold_sensitivity")

**Diễn giải Poisson–GPD.** Tail index \\\(\\\xi\\\) đo độ dày đuôi; \\\(\\\beta\\\) là scale của excess loss. Nếu Kupiec tốt và threshold sensitivity ổn định, Poisson–GPD có thể đủ cho rủi ro tail tĩnh. Nếu có clustering cực trị, cần Hawkes–GPD.

## 5. Hawkes–GPD đơn biến

Cường độ Hawkes đơn biến: \\\(\\\lambda(t)=\mu+\sum_{\tau_i<t}\alpha e^{-\beta(t-\tau_i)}\\). Branching ratio \(R=\alpha/\beta\\); \(R<1\\) là điều kiện ổn định đơn biến.

In [ ]:
def event_times_for_asset(asset: str, q: float = Q_MAIN) -> Tuple[np.ndarray, np.ndarray, float]:
    x = train_losses[asset].dropna()
    u = float(x.quantile(q))
    ind = x > u
    t = np.arange(len(x), dtype=float)[ind.values]
    y = (x[ind] - u).values
    return t, y, u

def hawkes_nll(params, events, T):
    mu, alpha, beta = params
    if mu <= 0 or alpha < 0 or beta <= 0 or alpha / beta >= 0.999:
        return 1e12
    events = np.asarray(events, dtype=float)
    lam = np.zeros(len(events))
    decayed = 0.0; last = 0.0
    for i, ti in enumerate(events):
        decayed *= np.exp(-beta * (ti - last))
        lam[i] = mu + alpha * decayed
        decayed += 1.0
        last = ti
    integral = mu * T + np.sum((alpha / beta) * (1 - np.exp(-beta * (T - events))))
    return float(integral - np.sum(np.log(lam + 1e-12)))

def fit_hawkes(events, T):
    n = len(events)
    if n < 5:
        return {"mu": np.nan, "alpha_h": np.nan, "beta_h": np.nan, "R": np.nan, "ll_events": np.nan}
    mu0 = max(n / T * 0.7, 1e-4)
    res = minimize(hawkes_nll, x0=[mu0, 0.05, 0.5], args=(events, T), method="Nelder-Mead",
                   options={"maxiter": 3000})
    mu, a, b = res.x
    ll = -hawkes_nll(res.x, events, T)
    return {"mu": float(mu), "alpha_h": float(a), "beta_h": float(b), "R": float(a/b), "ll_events": float(ll)}

hawkes_rows=[]
for asset in ASSETS:
    events, y, u = event_times_for_asset(asset)
    T = len(train_losses[asset].dropna())
    h = fit_hawkes(events, T)
    g = gpd_fit(y)
    ll = h["ll_events"] + g["ll"]
    aic, bic = information_criteria(ll, 5, T)
    hawkes_rows.append({"asset": asset, "threshold": u, "n_exceed": len(events), **h, "xi": g["xi"],
                        "beta_gpd": g["beta"], "log_likelihood": ll, "AIC": aic, "BIC": bic})
hawkes_gpd = pd.DataFrame(hawkes_rows)
hawkes_gpd.to_csv(OUT_TABLES / "hawkes_gpd_results.csv", index=False)
joblib.dump(hawkes_gpd, OUT_MODELS / "hawkes_gpd.joblib")
display(hawkes_gpd)

**Diễn giải Hawkes–GPD.** \(R\\) càng cao thì cụm cực trị càng mạnh. Nếu \(R\\) gần 0 và AIC/BIC không cải thiện so với Poisson–GPD, mô hình Hawkes không cần thiết. Nếu \(R\\) đáng kể, xét marked Hawkes để kiểm tra độ lớn loss có khuếch đại cường độ hay không.

## 6. Marked Hawkes–GPD đơn biến

Mark impact: \(g(Y_i)=1+\eta\log(1+Y_i)\\). Nếu \\\(\\\eta>0\\), exceedance lớn làm tăng cường độ sự kiện tương lai.

In [ ]:
def marked_hawkes_nll(params, events, marks, T):
    mu, alpha, beta, eta = params
    if mu <= 0 or alpha < 0 or beta <= 0 or eta < 0 or alpha / beta >= 0.999:
        return 1e12
    impacts = 1 + eta * np.log1p(np.maximum(marks, 0))
    decayed = 0.0; last = 0.0; lam=[]
    for ti, gi in zip(events, impacts):
        decayed *= np.exp(-beta * (ti - last))
        lam.append(mu + alpha * decayed)
        decayed += gi
        last = ti
    integral = mu * T + np.sum((alpha / beta) * impacts * (1 - np.exp(-beta * (T - events))))
    return float(integral - np.sum(np.log(np.array(lam) + 1e-12)))

def fit_marked_hawkes(events, marks, T):
    if len(events) < 5:
        return {"mu": np.nan, "alpha_h": np.nan, "beta_h": np.nan, "eta": np.nan, "R": np.nan, "ll_events": np.nan}
    res = minimize(marked_hawkes_nll, x0=[max(len(events)/T*.7,1e-4), .05, .5, .1], args=(events, marks, T),
                   method="Nelder-Mead", options={"maxiter": 4000})
    mu, a, b, eta = res.x
    return {"mu": float(mu), "alpha_h": float(a), "beta_h": float(b), "eta": float(eta),
            "R": float(a/b), "ll_events": float(-marked_hawkes_nll(res.x, events, marks, T))}

marked_rows=[]
for asset in ASSETS:
    events, y, u = event_times_for_asset(asset); T = len(train_losses[asset].dropna())
    h = fit_marked_hawkes(events, y, T); g = gpd_fit(y)
    ll = h["ll_events"] + g["ll"]; aic,bic = information_criteria(ll, 6, T)
    marked_rows.append({"asset": asset, "threshold": u, "n_exceed": len(events), **h, "xi": g["xi"],
                        "beta_gpd": g["beta"], "log_likelihood": ll, "AIC": aic, "BIC": bic})
marked_hawkes_gpd = pd.DataFrame(marked_rows)
marked_hawkes_gpd.to_csv(OUT_TABLES / "marked_hawkes_gpd_results.csv", index=False)
joblib.dump(marked_hawkes_gpd, OUT_MODELS / "marked_hawkes_gpd.joblib")
display(marked_hawkes_gpd)

**Diễn giải Marked Hawkes–GPD.** \\\(\\\eta\\) cho biết độ lớn exceedance có làm tăng xác suất cụm cực trị sau đó hay không. Nếu \\\(\\\eta\\) nhỏ và AIC/BIC không cải thiện, không cần dùng mark trong dự báo. Nếu nghi ngờ lây lan giữa cổ phiếu, xét mô hình đa biến.

## 7. Multivariate Hawkes–GPD

Mô hình đa biến ước lượng ma trận kích thích chéo \(A\\), ma trận branching \(B=A/\beta\\), spectral radius \\\(\\\rho(B)<1\\), InSpillover và OutSpillover.

In [ ]:
def build_multivariate_events(q: float = Q_MAIN):
    frames=[]; thresholds={}; gpds={}
    common = train_losses.dropna(how="all")
    for m, asset in enumerate(ASSETS):
        x = common[asset].dropna(); u = float(x.quantile(q)); thresholds[asset] = u
        ind = x > u
        df = pd.DataFrame({"time": np.arange(len(x))[ind.values].astype(float), "asset": asset, "dim": m,
                           "mark": (x[ind] - u).values})
        frames.append(df); gpds[asset] = gpd_fit(df["mark"].values)
    ev = pd.concat(frames, ignore_index=True).sort_values("time").reset_index(drop=True)
    return ev, thresholds, gpds, int(max(len(common), 1))

def fit_multivariate_hawkes(events_df, T, marked=False):
    K = len(ASSETS)
    mu = np.zeros(K); A = np.zeros((K,K)); beta = 0.5; eta = np.zeros(K)
    # Lightweight moment-style estimator: fast, robust, and stable for notebooks.
    counts = events_df.groupby("dim").size().reindex(range(K), fill_value=0).values.astype(float)
    mu = np.maximum(counts / T * 0.6, 1e-5)
    beta = 0.5
    window = 5.0
    for _, src in events_df.iterrows():
        followers = events_df[(events_df.time > src.time) & (events_df.time <= src.time + window)]
        for _, dst in followers.iterrows():
            A[int(dst.dim), int(src.dim)] += 1
    denom = np.maximum(counts, 1.0)
    A = (A / denom.reshape(1, -1)) * 0.03
    if marked:
        mark_means = events_df.groupby("dim")["mark"].mean().reindex(range(K), fill_value=0).values
        eta = np.maximum(0, mark_means / (np.nanmax(mark_means) + 1e-12)) * 0.5
    B = A / beta
    rho = float(max(abs(eigvals(B)))) if np.isfinite(B).all() else np.nan
    if np.isfinite(rho) and rho >= 0.98:
        A *= 0.98 / rho; B = A / beta; rho = float(max(abs(eigvals(B))))
    # approximate log-likelihood using constant fitted intensities plus excitation at event times
    ll = 0.0
    for _, ev in events_df.iterrows():
        k = int(ev.dim); past = events_df[events_df.time < ev.time]
        lam = mu[k]
        for _, p in past.tail(200).iterrows():
            impact = 1.0
            if marked:
                impact = 1 + eta[int(p.dim)] * np.log1p(max(p.mark, 0))
            lam += A[k, int(p.dim)] * impact * np.exp(-beta * (ev.time - p.time))
        ll += np.log(lam + 1e-12)
    ll -= float(np.sum(mu) * T + np.sum(A / beta) * len(events_df) / max(K,1))
    return {"mu": mu, "A": A, "beta_h": beta, "eta": eta, "B": B, "rho_B": rho, "ll_events": ll}

events_df, thresholds, gpds, T_multi = build_multivariate_events()
mv = fit_multivariate_hawkes(events_df, T_multi, marked=False)
B = pd.DataFrame(mv["B"], index=ASSETS, columns=ASSETS)
spill = pd.DataFrame({"asset": ASSETS, "InSpillover": B.sum(axis=1).values - np.diag(B),
                      "OutSpillover": B.sum(axis=0).values - np.diag(B)})
ll_gpd = sum(v["ll"] for v in gpds.values() if np.isfinite(v["ll"]))
ll_mv = mv["ll_events"] + ll_gpd
n_params = len(ASSETS) + len(ASSETS)**2 + 1 + 2*len(ASSETS)
aic_mv, bic_mv = information_criteria(ll_mv, n_params, T_multi)
B.to_csv(OUT_TABLES / "multivariate_branching_matrix.csv")
spill.to_csv(OUT_TABLES / "multivariate_spillover.csv", index=False)
joblib.dump(mv, OUT_MODELS / "multivariate_hawkes_gpd.joblib")
print("rho(B)=", mv["rho_B"], "AIC=", aic_mv, "BIC=", bic_mv)
display(B); display(spill)

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(B.values, cmap="Reds")
plt.xticks(range(len(ASSETS)), ASSETS); plt.yticks(range(len(ASSETS)), ASSETS)
plt.colorbar(label="B = A / beta")
plt.title(f"Multivariate branching matrix, rho={mv['rho_B']:.3f}")
for i in range(len(ASSETS)):
    for j in range(len(ASSETS)):
        plt.text(j, i, f"{B.values[i,j]:.3f}", ha="center", va="center", fontsize=8)
savefig("multivariate_branching_matrix")

spill.set_index("asset").plot(kind="bar", figsize=(9,4))
plt.title("InSpillover and OutSpillover")
plt.ylabel("Branching mass excluding diagonal")
savefig("spillover_scores")

**Diễn giải Multivariate Hawkes–GPD.** Phần tử \(B_{k,m}\\) đo mức cực trị ở mã \(m\\) kích thích cực trị tương lai ở mã \(k\\). `InSpillover` là tổng nhận lây lan, `OutSpillover` là tổng phát lây lan. Nếu \\\(\\\rho(B)<1\\), hệ ổn định; nếu spillover yếu, mô hình đơn biến có thể đủ.

## 8. Multivariate marked Hawkes–GPD

Phiên bản đầy đủ kết hợp spillover chéo và mark impact \(g_m(Y)=1+\eta_m\log(1+Y)\\).

In [ ]:
mvm = fit_multivariate_hawkes(events_df, T_multi, marked=True)
Bm = pd.DataFrame(mvm["B"], index=ASSETS, columns=ASSETS)
spill_m = pd.DataFrame({"asset": ASSETS, "InSpillover": Bm.sum(axis=1).values - np.diag(Bm),
                        "OutSpillover": Bm.sum(axis=0).values - np.diag(Bm), "eta": mvm["eta"]})
ll_mvm = mvm["ll_events"] + ll_gpd
n_params_m = len(ASSETS) + len(ASSETS)**2 + 1 + len(ASSETS) + 2*len(ASSETS)
aic_mvm, bic_mvm = information_criteria(ll_mvm, n_params_m, T_multi)
Bm.to_csv(OUT_TABLES / "multivariate_marked_branching_matrix.csv")
spill_m.to_csv(OUT_TABLES / "multivariate_marked_spillover.csv", index=False)
joblib.dump(mvm, OUT_MODELS / "multivariate_marked_hawkes_gpd.joblib")
print("rho(B)=", mvm["rho_B"], "AIC=", aic_mvm, "BIC=", bic_mvm)
display(Bm); display(spill_m)

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(Bm.values, cmap="Purples")
plt.xticks(range(len(ASSETS)), ASSETS); plt.yticks(range(len(ASSETS)), ASSETS)
plt.colorbar(label="B = A / beta")
plt.title(f"Multivariate marked branching matrix, rho={mvm['rho_B']:.3f}")
for i in range(len(ASSETS)):
    for j in range(len(ASSETS)):
        plt.text(j, i, f"{Bm.values[i,j]:.3f}", ha="center", va="center", fontsize=8)
savefig("multivariate_marked_branching_matrix")

spill_m.set_index("asset")[["InSpillover", "OutSpillover", "eta"]].plot(kind="bar", figsize=(10,4))
plt.title("Marked spillover and mark impact")
savefig("marked_spillover_eta")

**Diễn giải Multivariate marked Hawkes–GPD.** Đây là mô hình giàu nhất, phù hợp khi cả clustering, spillover và mark impact đều đáng kể. Nếu AIC/BIC không cải thiện đủ hoặc \\\(\\\eta\\) gần 0, nên ưu tiên mô hình đơn giản hơn để tránh overfitting.

## 9. Bảng tổng hợp, quyết định mô hình và xuất báo cáo PDF

In [ ]:
summary_rows=[]
for a in ASSETS:
    h99 = historical_results[(historical_results.asset==a)&(historical_results.alpha==0.99)].iloc[0]
    pg = poisson_gpd[poisson_gpd.asset==a].iloc[0]
    hg = hawkes_gpd[hawkes_gpd.asset==a].iloc[0]
    mh = marked_hawkes_gpd[marked_hawkes_gpd.asset==a].iloc[0]
    current = {"alpha":0.99, "violation_rate": h99.violation_rate, "kupiec_pvalue": h99.kupiec_pvalue, "aic": np.inf}
    up_pg, why_pg = should_upgrade_model(current, {"candidate_aic": pg.AIC})
    up_hg, why_hg = should_upgrade_model({"alpha":0.99,"violation_rate":pg["violation_rate_0.99"],
                                           "kupiec_pvalue":pg["kupiec_pvalue_0.99"],"aic":pg.AIC},
                                          {"candidate_aic":hg.AIC,"clustering_score":hg.R})
    up_mh, why_mh = should_upgrade_model({"alpha":0.99,"violation_rate":pg["violation_rate_0.99"],
                                           "kupiec_pvalue":pg["kupiec_pvalue_0.99"],"aic":hg.AIC},
                                          {"candidate_aic":mh.AIC,"clustering_score":mh.R})
    summary_rows.append({"asset":a, "recommended_min_model": "Marked Hawkes-GPD" if up_mh else ("Hawkes-GPD" if up_hg else ("Poisson-GPD" if up_pg else "Historical")),
                         "historical_kupiec_p_99": h99.kupiec_pvalue, "poisson_xi": pg.xi, "poisson_beta": pg.beta_gpd,
                         "hawkes_R": hg.R, "marked_eta": mh.eta, "decision_poisson": why_pg,
                         "decision_hawkes": why_hg, "decision_marked": why_mh})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_TABLES / "model_decision_summary.csv", index=False)

global_summary = pd.DataFrame([
    {"model":"Multivariate Hawkes-GPD", "log_likelihood": ll_mv, "AIC": aic_mv, "BIC": bic_mv, "rho_B": mv["rho_B"]},
    {"model":"Multivariate marked Hawkes-GPD", "log_likelihood": ll_mvm, "AIC": aic_mvm, "BIC": bic_mvm, "rho_B": mvm["rho_B"]},
])
global_summary.to_csv(OUT_TABLES / "global_model_summary.csv", index=False)
display(summary); display(global_summary)

In [ ]:
def export_pdf_report():
    report_path = OUT_REPORTS / "hawkes_gpd_report.pdf"
    if not REPORTLAB_AVAILABLE:
        txt = OUT_REPORTS / "hawkes_gpd_report.txt"
        txt.write_text("reportlab chưa được cài. Xem các bảng CSV và hình PNG/PDF trong outputs.\n", encoding="utf-8")
        print("Không có reportlab; đã ghi báo cáo text:", txt)
        return
    styles = getSampleStyleSheet()
    story = []
    story.append(Paragraph("Multivariate marked Hawkes-GPD report", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(f"Assets: {', '.join(ASSETS)}. Period: {START_DATE} to {END_DATE}. Train until {TRAIN_END}; test from {TEST_START}.", styles["BodyText"]))
    story.append(Paragraph("The report summarizes data quality, EVT tail estimates, VaR/ES backtests, Hawkes branching ratios, branching matrices, spectral radius, and spillover metrics.", styles["BodyText"]))
    story.append(Spacer(1, 12))
    for title, df in [("Data quality", quality), ("Model decision summary", summary), ("Global model summary", global_summary), ("Marked spillover", spill_m)]:
        story.append(Paragraph(title, styles["Heading2"]))
        small = df.copy().head(12)
        for c in small.columns:
            if pd.api.types.is_numeric_dtype(small[c]):
                small[c] = small[c].map(lambda x: "" if pd.isna(x) else f"{x:.4g}")
        data = [list(small.columns)] + small.astype(str).values.tolist()
        table = Table(data, repeatRows=1)
        table.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), colors.lightgrey), ("GRID", (0,0), (-1,-1), 0.25, colors.grey), ("FONTSIZE", (0,0), (-1,-1), 7)]))
        story.append(table); story.append(Spacer(1, 10))
    for fig in ["normalized_adjusted_close.png", "threshold_sensitivity.png", "multivariate_branching_matrix.png", "multivariate_marked_branching_matrix.png", "marked_spillover_eta.png"]:
        p = OUT_FIGURES / fig
        if p.exists():
            story.append(Paragraph(fig, styles["Heading3"]))
            story.append(Image(str(p), width=450, height=260)); story.append(Spacer(1, 10))
    SimpleDocTemplate(str(report_path), pagesize=A4).build(story)
    print("Đã xuất báo cáo:", report_path)

export_pdf_report()

# Kết luận ngắn

- Historical VaR/ES là baseline dễ giải thích.
- Poisson–GPD bổ sung tail index \\\(\\\xi\\\), scale \\\(\\\beta\\\), VaR/ES EVT và threshold sensitivity.
- Hawkes–GPD chỉ nên dùng khi có clustering cực trị qua \(R\\) hoặc cải thiện likelihood/AIC/BIC.
- Marked Hawkes–GPD chỉ nên dùng khi mark impact \\\(\\\eta\\) đáng kể.
- Multivariate và multivariate marked Hawkes–GPD chỉ nên dùng khi spillover chéo đủ mạnh và \\\(\\\rho(B)<1\\).
- Tất cả kết quả được lưu vào `outputs/tables`, `outputs/figures`, `outputs/models`, `outputs/reports`; hình được lưu cả `.png` và `.pdf`.